# 9.15 — Preference Optimization (DPO, IPO)

Preference optimization trains a language model from chosen-versus-rejected responses without running a separate reinforcement-learning loop. In this lesson, we reduce DPO and IPO to tiny NumPy arrays: log-probabilities, reference margins, sigmoid losses, implicit rewards, and preference-pair training curves you can inspect cell by cell.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build preference optimization one idea at a time. Run each cell in order and read the printed intermediate values — the point is to see exactly which log-probability term controls behavior, why the reference model matters, and how DPO and IPO turn a pairwise preference into a trainable scalar loss. This walkthrough is self-contained and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, exponentials, and tiny optimization loops.
import matplotlib.pyplot as plt  # all walkthrough visualizations.
np.random.seed(0)  # reproducibility for toy preference data.

### 1. Preference pairs as log-probability comparisons

Preference data does not say "the chosen answer is absolutely good." It says that, for the same prompt, one response was preferred to another. We therefore start with four log-probabilities: the trainable policy's log-probability of the chosen and rejected responses, and the frozen reference policy's same two values. Log-probabilities are negative because probabilities are below 1; differences are easier to optimize than raw probabilities.

In [ ]:
logp_chosen_w = -1.0   # log pi_theta(y+ | x): trainable policy on the chosen response.
logp_reject_w = -2.0   # log pi_theta(y- | x): trainable policy on the rejected response.
ref_chosen_w = -1.5    # log pi_0(y+ | x): frozen reference model on the chosen response.
ref_reject_w = -2.2    # log pi_0(y- | x): frozen reference model on the rejected response.
print("policy logs:", logp_chosen_w, logp_reject_w)
print("reference logs:", ref_chosen_w, ref_reject_w)

▶ What you'll see: four negative numbers, two from the model being trained and two from the frozen reference.

In [ ]:
plt.figure(figsize=(4.6, 3))
plt.bar(["πθ chosen", "πθ rejected", "π0 chosen", "π0 rejected"],
        [logp_chosen_w, logp_reject_w, ref_chosen_w, ref_reject_w], color=["teal", "orange", "gray", "lightgray"])
plt.axhline(0, color="black", linewidth=0.8)
plt.ylabel("log probability")
plt.title("1: preference pair log-probabilities")
plt.xticks(rotation=20)
plt.show()

▶ What you'll see: the chosen response has higher log-probability than the rejected response under both models.

*Why it's done this way:* a preference pair gives only relative information, so the training signal must compare two completions for the same prompt. Log-probabilities turn products over tokens into sums, and keeping the reference values beside the trainable values lets us ask whether the new policy improved the chosen-vs-rejected odds beyond what the old model already believed.

### 2. The DPO margin and loss

DPO first computes how much the trainable policy prefers the chosen response over the rejected response **relative to the reference**. The chosen margin is $\log\pi_\theta^+ - \log\pi_0^+$, the rejected margin is $\log\pi_\theta^- - \log\pi_0^-$, and the DPO gap is their difference. Passing $\beta\cdot\text{gap}$ through a sigmoid turns the pair into a binary-classification probability that the chosen response should win.

In [ ]:
chosen_margin_w = logp_chosen_w - ref_chosen_w     # -1.0 - (-1.5) = 0.5.
reject_margin_w = logp_reject_w - ref_reject_w     # -2.0 - (-2.2) = 0.2.
gap_w = chosen_margin_w - reject_margin_w          # DPO's preference logit before beta.
print("chosen margin:", round(chosen_margin_w, 3))
print("rejected margin:", round(reject_margin_w, 3))
print("DPO gap:", round(gap_w, 3))
assert round(gap_w, 3) == 0.300

▶ What you'll see: the policy improved the chosen response by 0.5 nats and the rejected response by 0.2 nats, so the preference gap is 0.3.

In [ ]:
beta_w = 2.0
logit_w = beta_w * gap_w
prob_w = 1 / (1 + np.exp(-logit_w))
loss_w = -np.log(prob_w)
print("beta * gap:", round(logit_w, 3))
print("sigmoid probability:", round(prob_w, 3))
print("DPO loss:", round(loss_w, 3))
assert round(logit_w, 3) == 0.600 and round(loss_w, 3) == 0.437

▶ What you'll see: the margin becomes a win probability of about 0.646 and a loss of about 0.437.

In [ ]:
gaps_grid_w = np.linspace(-2, 2, 100)
loss_grid_w = -np.log(1 / (1 + np.exp(-beta_w * gaps_grid_w)))
plt.figure(figsize=(4.6, 3))
plt.plot(gaps_grid_w, loss_grid_w, color="purple")
plt.scatter([gap_w], [loss_w], color="red")
plt.axvline(0, color="black", linewidth=0.8)
plt.xlabel("DPO gap")
plt.ylabel("-log sigmoid(beta gap)")
plt.title("2: DPO loss rewards positive gaps")
plt.show()

▶ What you'll see: loss is high when the rejected response wins and falls as the chosen response wins by a larger margin.

*Why it's done this way:* binary cross-entropy is the natural loss for "chosen should beat rejected." The subtraction against the reference is the key DPO trick: it does not simply maximize chosen likelihood; it maximizes the **change in chosen-vs-rejected odds** relative to a stable anchor, which is what connects the objective to KL-regularized reward optimization.

### 3. Beta controls preference pressure and saturation

The scalar $\beta$ multiplies the DPO gap before the sigmoid. A small $\beta$ makes the preference label soft; a large $\beta$ makes the same gap look more decisive. This is useful, but too large a value saturates the sigmoid: probabilities become close to 0 or 1, and gradients stop being informative.

In [ ]:
betas_w = np.array([0.5, 1.0, 2.0, 5.0])
probs_w = 1 / (1 + np.exp(-betas_w * gap_w))
losses_beta_w = -np.log(probs_w)
print("betas:", betas_w)
print("win probabilities:", np.round(probs_w, 3))
assert round(float(probs_w[1]), 3) == 0.574 and round(float(probs_w[2]), 3) == 0.646

▶ What you'll see: doubling beta from 1 to 2 changes the probability from 0.574 to 0.646 for the same 0.3 gap.

In [ ]:
grad_mag_w = beta_w * (1 - prob_w)  # magnitude of d loss / d gap for positive label.
print("gradient magnitude at beta=2:", round(grad_mag_w, 3))
plt.figure(figsize=(4.6, 3))
plt.plot(betas_w, probs_w, marker="o", label="sigmoid(beta gap)")
plt.plot(betas_w, losses_beta_w, marker="s", label="loss")
plt.xlabel("beta")
plt.title("3: beta changes preference pressure")
plt.legend()
plt.show()

▶ What you'll see: higher beta pushes probability up and loss down for this positive gap, but the curve starts to flatten.

*Why it's done this way:* $\beta$ is a temperature-like knob for the KL tradeoff. Increasing it says "push harder on preferences," but the logistic derivative is largest near zero and small in the tails. A huge $\beta$ can make already-correct pairs look solved, leaving weak gradients and brittle updates.

### 4. Implicit reward from a policy/reference ratio

DPO never trains a separate reward model, but it still has an implicit reward: $r_\theta(x,y)=\beta(\log\pi_\theta(y|x)-\log\pi_0(y|x))$ up to an additive constant. If the trainable policy assigns a response more probability than the reference does, that response receives positive implicit reward; if it assigns less, the reward is negative.

In [ ]:
reward_chosen_w = beta_w * chosen_margin_w
reward_reject_w = beta_w * reject_margin_w
reward_gap_w = reward_chosen_w - reward_reject_w
print("implicit rewards:", round(reward_chosen_w, 3), round(reward_reject_w, 3))
print("reward gap:", round(reward_gap_w, 3))
assert round(reward_gap_w, 3) == 0.600

▶ What you'll see: the chosen response has implicit reward 1.0, the rejected response 0.4, and their gap is exactly the DPO logit.

In [ ]:
plt.figure(figsize=(4.4, 3))
plt.bar(["chosen", "rejected", "gap"], [reward_chosen_w, reward_reject_w, reward_gap_w], color=["teal", "orange", "purple"])
plt.title("4: implicit rewards from policy/reference ratios")
plt.ylabel("beta · (log πθ - log π0)")
plt.show()

▶ What you'll see: DPO's binary logit is just the difference between two implicit rewards.

*Why it's done this way:* the reference ratio is a built-in KL anchor. It rewards the trainable model for moving probability mass toward preferred responses, but only relative to what the starting model already did. That is why dropping the reference terms changes the problem: the model would chase raw likelihood rather than controlled improvement over the base policy.

### 5. IPO as a target-gap regression

IPO uses the same pairwise gap but does not drive the sigmoid logit toward infinity. Instead, it asks the gap to land near a finite target, commonly $1/(2\beta)$ in this simplified scalar view. That turns the preference update into squared error: enough separation to encode the preference, but not unlimited separation.

In [ ]:
ipo_target_w = 1 / (2 * beta_w)
ipo_loss_w = (gap_w - ipo_target_w) ** 2
print("IPO target gap:", round(ipo_target_w, 3))
print("actual gap:", round(gap_w, 3))
print("IPO squared error:", round(ipo_loss_w, 4))
assert round(ipo_target_w, 3) == 0.250 and round(ipo_loss_w, 4) == 0.0025

▶ What you'll see: beta=2 gives a target gap of 0.25; the actual 0.3 gap is close, so the IPO loss is tiny.

In [ ]:
gap_grid_ipo_w = np.linspace(-0.3, 0.9, 100)
ipo_curve_w = (gap_grid_ipo_w - ipo_target_w) ** 2
plt.figure(figsize=(4.6, 3))
plt.plot(gap_grid_ipo_w, ipo_curve_w, color="darkorange")
plt.axvline(ipo_target_w, color="black", linestyle="--", label="target")
plt.scatter([gap_w], [ipo_loss_w], color="red", label="actual")
plt.xlabel("policy-reference preference gap")
plt.ylabel("IPO squared error")
plt.title("5: IPO prefers a finite gap")
plt.legend()
plt.show()

▶ What you'll see: a parabola with its minimum at 0.25, not at an infinitely large chosen-over-rejected gap.

*Why it's done this way:* DPO's logistic loss keeps rewarding larger margins, though with diminishing returns. IPO makes the desired margin explicit: the preference pair should separate chosen from rejected by a controlled amount. That can reduce overconfident preference fitting when labels are noisy or the reference anchor matters strongly.

### 6. One toy DPO training loop

To see optimization, we give each response a single scalar feature and let one parameter $\theta$ change the policy log-probabilities. The reference log-probabilities stay fixed. Each update computes the DPO gap for every pair, takes the average loss, and moves $\theta$ in the direction that makes chosen responses win more often.

In [ ]:
chosen_feat_w = np.array([1.0, 0.8, 1.2, 0.6])
reject_feat_w = np.array([0.1, 0.3, 0.2, 0.4])
base_chosen_w = np.array([-1.4, -1.2, -1.8, -1.0])
base_reject_w = np.array([-1.3, -1.1, -1.6, -1.1])
ref_chosen_vec_w = base_chosen_w.copy()
ref_reject_vec_w = base_reject_w.copy()
print("feature gaps:", chosen_feat_w - reject_feat_w)

▶ What you'll see: chosen responses have larger features than rejected responses, so a positive theta should help.

In [ ]:
theta_w = 0.0
eta_w = 0.5
loss_hist_w = []
for step_w in range(40):
    logp_c_w = base_chosen_w + theta_w * chosen_feat_w
    logp_r_w = base_reject_w + theta_w * reject_feat_w
    gap_vec_w = (logp_c_w - ref_chosen_vec_w) - (logp_r_w - ref_reject_vec_w)
    prob_vec_w = 1 / (1 + np.exp(-beta_w * gap_vec_w))
    loss_hist_w.append(float(np.mean(-np.log(prob_vec_w))))
    grad_theta_w = np.mean(-beta_w * (1 - prob_vec_w) * (chosen_feat_w - reject_feat_w))
    theta_w -= eta_w * grad_theta_w
print("final theta:", round(theta_w, 3), "loss start -> end:", round(loss_hist_w[0], 3), "->", round(loss_hist_w[-1], 3))
assert loss_hist_w[-1] < loss_hist_w[0]

▶ What you'll see: theta increases and the DPO loss decreases because the chosen features are favored.

In [ ]:
plt.figure(figsize=(4.6, 3))
plt.plot(loss_hist_w, color="teal")
plt.xlabel("step")
plt.ylabel("mean DPO loss")
plt.title("6: toy DPO training decreases loss")
plt.show()

▶ What you'll see: a smooth downward curve, showing that even a one-parameter policy can learn from preference pairs.

*Why it's done this way:* the derivative of $-\log\sigma(\beta g)$ with respect to the gap is $-\beta(1-\sigma(\beta g))$. If chosen features exceed rejected features, increasing $\theta$ increases the gap, so gradient descent raises $\theta$ until the pairwise classification loss flattens.

### 7. Preference data quality and the label-noise pitfall

DPO and IPO simplify optimization, not data collection. If preference labels are noisy, the loss faithfully trains on that noise. A flipped pair says the rejected response should win; the math cannot know it was mislabeled unless validation, audits, or better data catch it.

In [ ]:
true_gaps_w = np.array([1.2, 0.9, 0.7, 0.4, 0.8])
labels_clean_w = np.ones_like(true_gaps_w)
labels_noisy_w = np.array([1, 1, 1, 1, -1])  # last pair is flipped: rejected is incorrectly treated as chosen.
clean_loss_w = -np.log(1 / (1 + np.exp(-true_gaps_w)))
noisy_loss_w = -np.log(1 / (1 + np.exp(-(labels_noisy_w * true_gaps_w))))
print("clean mean loss:", round(float(np.mean(clean_loss_w)), 3))
print("noisy mean loss:", round(float(np.mean(noisy_loss_w)), 3))

▶ What you'll see: the mislabeled pair changes the objective because its signed gap points the other way.

In [ ]:
plt.figure(figsize=(4.6, 3))
plt.bar(np.arange(len(true_gaps_w)) - 0.18, clean_loss_w, width=0.36, label="clean", color="teal")
plt.bar(np.arange(len(true_gaps_w)) + 0.18, noisy_loss_w, width=0.36, label="with flip", color="crimson")
plt.xlabel("preference pair")
plt.ylabel("pair loss")
plt.title("7: label noise changes the training signal")
plt.legend()
plt.show()

▶ What you'll see: one flipped label creates a different loss contribution, pushing the policy in the wrong direction for that pair.

*Why it's done this way:* preference optimization is only as aligned as the comparisons it receives. The formula optimizes the provided ranking exactly; it does not verify truth, helpfulness, or safety by itself. That is why held-out preference evaluation and label-quality checks remain part of the system, even when the training loss is elegant.

## ✍️ Toy Examples

> ✍️ **Toy examples — trace each mechanic by hand.** Separate from the walkthrough above, here
> is one tiny, fully hand-traceable toy per computational mechanic in this lesson. Each uses a
> handful of small numbers, prints every intermediate value with an inline `# ->` showing the
> result, draws one picture, and ends with an `assert` that pins the answer.

### ✍️ Toy 1 · Preference pairs are log-probability margins

A chosen-versus-rejected pair becomes two margins: one under the trainable policy and one under the
reference. DPO uses their difference.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


t1_rng = np.random.default_rng(0)                     # seeded generator for reproducibility.
t1_policy_logs = np.array([-0.8, -1.7])               # -> [-0.8, -1.7]
print("policy log-probs [chosen, rejected]:", t1_policy_logs.tolist())  # -> [-0.8, -1.7]
t1_ref_logs = np.array([-1.0, -1.5])                  # -> [-1.0, -1.5]
print("reference log-probs [chosen, rejected]:", t1_ref_logs.tolist())  # -> [-1.0, -1.5]
t1_policy_margin = float(t1_policy_logs[0] - t1_policy_logs[1])  # -> 0.9
print("policy chosen-minus-rejected margin:", round(t1_policy_margin, 3))  # -> 0.9
t1_ref_margin = float(t1_ref_logs[0] - t1_ref_logs[1])  # -> 0.5
print("reference chosen-minus-rejected margin:", round(t1_ref_margin, 3))  # -> 0.5
t1_gap = t1_policy_margin - t1_ref_margin             # -> 0.4
print("DPO gap:", round(t1_gap, 3))                   # -> 0.4
assert round(t1_gap, 3) == 0.4

plt.figure(figsize=(4.8, 2.8))
plt.bar(["π chosen", "π rejected", "π₀ chosen", "π₀ rejected"], [*t1_policy_logs, *t1_ref_logs], color=["#4c78a8", "#f28e2b", "gray", "lightgray"])
plt.axhline(0, color="black", linewidth=0.8)
plt.ylabel("log probability")
plt.title("Toy 1 · pairwise log-probs")
plt.xticks(rotation=20)
plt.show()

▶ What you'll see: the policy margin exceeds the reference margin by `0.4`, giving DPO a positive gap.

### ✍️ Toy 2 · DPO loss is negative log-sigmoid

DPO turns the gap into a binary preference probability with `sigmoid(beta × gap)`, then minimizes the
negative log of that probability.

In [ ]:
import numpy as np


t2_rng = np.random.default_rng(0)                     # seeded generator for reproducibility.
t2_gap = 0.4                                          # -> 0.4
print("DPO gap:", t2_gap)                             # -> 0.4
t2_beta = 2.0                                         # -> 2.0
print("beta:", t2_beta)                               # -> 2.0
t2_logit = t2_beta * t2_gap                           # -> 0.8
print("beta times gap:", round(t2_logit, 3))          # -> 0.8
t2_prob = 1 / (1 + np.exp(-t2_logit))                 # -> 0.69
print("chosen-win probability:", round(float(t2_prob), 3))  # -> 0.69
t2_loss = -np.log(t2_prob)                            # -> 0.371
print("DPO loss:", round(float(t2_loss), 3))          # -> 0.371
assert round(float(t2_loss), 3) == 0.371

plt.figure(figsize=(4.8, 2.8))
t2_grid = np.linspace(-1.0, 1.0, 80)
t2_curve = -np.log(1 / (1 + np.exp(-t2_beta * t2_grid)))
plt.plot(t2_grid, t2_curve, color="#4c78a8")
plt.scatter([t2_gap], [t2_loss], color="#e15759")
plt.axvline(0, color="black", linewidth=0.8)
plt.xlabel("DPO gap")
plt.ylabel("loss")
plt.title("Toy 2 · -log sigmoid")
plt.show()

▶ What you'll see: a positive gap sits on the low-loss side of the logistic curve.

### ✍️ Toy 3 · Beta changes pressure and saturation

The same gap looks soft at small beta and nearly solved at very large beta. The remaining error mass
`1 - sigmoid(beta × gap)` shrinks toward zero.

In [ ]:
import numpy as np


t3_rng = np.random.default_rng(0)                     # seeded generator for reproducibility.
t3_gap = 0.8                                          # -> 0.8
print("fixed gap:", t3_gap)                           # -> 0.8
t3_betas = np.array([0.5, 2.0, 5.0, 10.0])            # -> [0.5, 2.0, 5.0, 10.0]
print("betas:", t3_betas.tolist())                    # -> [0.5, 2.0, 5.0, 10.0]
t3_logits = t3_betas * t3_gap                         # -> [0.4, 1.6, 4.0, 8.0]
print("beta-gap logits:", np.round(t3_logits, 3).tolist())  # -> [0.4, 1.6, 4.0, 8.0]
t3_probs = 1 / (1 + np.exp(-t3_logits))               # -> [0.599, 0.832, 0.982, 1.0]
print("win probabilities:", np.round(t3_probs, 3).tolist())  # -> [0.599, 0.832, 0.982, 1.0]
t3_missing = 1 - t3_probs                             # -> [0.4013, 0.168, 0.018, 0.0003]
print("remaining error mass:", np.round(t3_missing, 4).tolist())  # -> [0.4013, 0.168, 0.018, 0.0003]
assert t3_missing[-1] < 0.001

plt.figure(figsize=(4.8, 2.8))
plt.plot(t3_betas, t3_probs, marker="o", label="probability")
plt.plot(t3_betas, t3_missing, marker="s", label="1 - probability")
plt.xlabel("beta")
plt.title("Toy 3 · beta saturation")
plt.legend()
plt.show()

▶ What you'll see: beta `10` makes the preference probability round to `1.0`, leaving almost no error mass.

### ✍️ Toy 4 · Implicit reward is a policy/reference ratio

DPO's implicit reward for one response is beta times how much the policy's log-probability exceeds
the reference's log-probability.

In [ ]:
import numpy as np


t4_rng = np.random.default_rng(0)                     # seeded generator for reproducibility.
t4_policy_logs = np.array([-0.8, -1.7, -2.2])         # -> [-0.8, -1.7, -2.2]
print("policy logs:", t4_policy_logs.tolist())        # -> [-0.8, -1.7, -2.2]
t4_ref_logs = np.array([-1.0, -1.5, -2.4])            # -> [-1.0, -1.5, -2.4]
print("reference logs:", t4_ref_logs.tolist())        # -> [-1.0, -1.5, -2.4]
t4_beta = 2.0                                         # -> 2.0
print("beta:", t4_beta)                               # -> 2.0
t4_margins = t4_policy_logs - t4_ref_logs             # -> [0.2, -0.2, 0.2]
print("log policy/reference margins:", np.round(t4_margins, 3).tolist())  # -> [0.2, -0.2, 0.2]
t4_rewards = t4_beta * t4_margins                     # -> [0.4, -0.4, 0.4]
print("implicit rewards:", np.round(t4_rewards, 3).tolist())  # -> [0.4, -0.4, 0.4]
t4_reward_gap = float(t4_rewards[0] - t4_rewards[1])  # -> 0.8
print("chosen-minus-rejected reward gap:", round(t4_reward_gap, 3))  # -> 0.8
assert round(t4_reward_gap, 3) == 0.8

plt.figure(figsize=(4.8, 2.8))
plt.bar(["chosen", "rejected", "other"], t4_rewards, color=["#59a14f", "#e15759", "#4c78a8"])
plt.axhline(0, color="black", linewidth=0.8)
plt.ylabel("implicit reward")
plt.title("Toy 4 · beta log-ratio rewards")
plt.show()

▶ What you'll see: the rejected response has negative implicit reward because the policy rates it below the reference.

### ✍️ Toy 5 · IPO targets a finite preference gap

IPO does not ask the pairwise gap to grow forever. It picks a finite target and penalizes squared
distance from that target.

In [ ]:
import numpy as np


t5_rng = np.random.default_rng(0)                     # seeded generator for reproducibility.
t5_beta = 2.0                                         # -> 2.0
print("beta:", t5_beta)                               # -> 2.0
t5_target = 1 / (2 * t5_beta)                         # -> 0.25
print("IPO target gap:", round(t5_target, 3))         # -> 0.25
t5_gaps = np.array([0.0, 0.25, 0.6])                  # -> [0.0, 0.25, 0.6]
print("candidate gaps:", t5_gaps.tolist())            # -> [0.0, 0.25, 0.6]
t5_losses = (t5_gaps - t5_target) ** 2                # -> [0.0625, 0.0, 0.1225]
print("IPO squared losses:", np.round(t5_losses, 4).tolist())  # -> [0.0625, 0.0, 0.1225]
t5_best_index = int(np.argmin(t5_losses))             # -> 1
print("best gap index:", t5_best_index)               # -> 1
assert t5_best_index == 1 and t5_losses[1] == 0.0

plt.figure(figsize=(4.8, 2.8))
t5_grid = np.linspace(-0.1, 0.8, 80)
t5_curve = (t5_grid - t5_target) ** 2
plt.plot(t5_grid, t5_curve, color="#f28e2b")
plt.scatter(t5_gaps, t5_losses, color="#4c78a8")
plt.axvline(t5_target, color="black", linestyle="--")
plt.xlabel("gap")
plt.ylabel("IPO loss")
plt.title("Toy 5 · finite target gap")
plt.show()

▶ What you'll see: the parabola bottoms out at gap `0.25`, not at an infinitely large margin.

### ✍️ Toy 6 · One scalar DPO update increases chosen gaps

With one scalar parameter, the DPO gradient points toward making chosen features outrank rejected
features. One step lowers the mean DPO loss.

In [ ]:
import numpy as np


t6_rng = np.random.default_rng(0)                     # seeded generator for reproducibility.
t6_theta = 0.0                                        # -> 0.0
print("initial theta:", t6_theta)                     # -> 0.0
t6_beta = 1.0                                         # -> 1.0
print("beta:", t6_beta)                               # -> 1.0
t6_feature_gap = np.array([0.8, 0.4])                 # -> [0.8, 0.4]
print("chosen-rejected feature gaps:", t6_feature_gap.tolist())  # -> [0.8, 0.4]
t6_gap = t6_theta * t6_feature_gap                    # -> [0.0, 0.0]
print("initial DPO gaps:", np.round(t6_gap, 3).tolist())  # -> [0.0, 0.0]
t6_prob = 1 / (1 + np.exp(-t6_beta * t6_gap))         # -> [0.5, 0.5]
print("initial probabilities:", np.round(t6_prob, 3).tolist())  # -> [0.5, 0.5]
t6_loss = float(np.mean(-np.log(t6_prob)))            # -> 0.693
print("initial mean loss:", round(t6_loss, 3))        # -> 0.693
t6_grad = float(np.mean(-t6_beta * (1 - t6_prob) * t6_feature_gap))  # -> -0.3
print("theta gradient:", round(t6_grad, 3))           # -> -0.3
t6_lr = 1.0                                           # -> 1.0
print("learning rate:", t6_lr)                        # -> 1.0
t6_theta_new = t6_theta - t6_lr * t6_grad             # -> 0.3
print("updated theta:", round(t6_theta_new, 3))       # -> 0.3
t6_gap_new = t6_theta_new * t6_feature_gap            # -> [0.24, 0.12]
print("new DPO gaps:", np.round(t6_gap_new, 3).tolist())  # -> [0.24, 0.12]
t6_prob_new = 1 / (1 + np.exp(-t6_beta * t6_gap_new)) # -> [0.56, 0.53]
print("new probabilities:", np.round(t6_prob_new, 3).tolist())  # -> [0.56, 0.53]
t6_loss_new = float(np.mean(-np.log(t6_prob_new)))    # -> 0.608
print("new mean loss:", round(t6_loss_new, 3))        # -> 0.608
assert t6_loss_new < t6_loss

plt.figure(figsize=(4.8, 2.8))
plt.bar(["before", "after"], [t6_loss, t6_loss_new], color=["gray", "#59a14f"])
plt.ylabel("mean DPO loss")
plt.title("Toy 6 · one DPO step")
plt.show()

▶ What you'll see: the update makes both gaps positive and lowers the mean loss from `0.693` to `0.608`.

### ✍️ Toy 7 · Flipped labels reverse one training signal

If a preference label is flipped, the signed gap changes sign for that pair. The same confident gap
then becomes a large loss instead of a small one.

In [ ]:
import numpy as np


t7_rng = np.random.default_rng(0)                     # seeded generator for reproducibility.
t7_true_gaps = np.array([1.0, 0.6, 0.8])              # -> [1.0, 0.6, 0.8]
print("true chosen-over-rejected gaps:", t7_true_gaps.tolist())  # -> [1.0, 0.6, 0.8]
t7_clean_labels = np.ones(3)                          # -> [1.0, 1.0, 1.0]
print("clean labels:", t7_clean_labels.tolist())      # -> [1.0, 1.0, 1.0]
t7_noisy_labels = np.array([1.0, 1.0, -1.0])          # -> [1.0, 1.0, -1.0]
print("noisy labels:", t7_noisy_labels.tolist())      # -> [1.0, 1.0, -1.0]
t7_clean_signed = t7_clean_labels * t7_true_gaps      # -> [1.0, 0.6, 0.8]
print("clean signed gaps:", t7_clean_signed.tolist()) # -> [1.0, 0.6, 0.8]
t7_noisy_signed = t7_noisy_labels * t7_true_gaps      # -> [1.0, 0.6, -0.8]
print("noisy signed gaps:", t7_noisy_signed.tolist()) # -> [1.0, 0.6, -0.8]
t7_clean_loss = -np.log(1 / (1 + np.exp(-t7_clean_signed)))  # -> [0.313, 0.437, 0.371]
print("clean losses:", np.round(t7_clean_loss, 3).tolist())  # -> [0.313, 0.437, 0.371]
t7_noisy_loss = -np.log(1 / (1 + np.exp(-t7_noisy_signed)))  # -> [0.313, 0.437, 1.171]
print("noisy losses:", np.round(t7_noisy_loss, 3).tolist())  # -> [0.313, 0.437, 1.171]
t7_loss_gap = t7_noisy_loss - t7_clean_loss           # -> [0.0, 0.0, 0.8]
print("extra loss from noise:", np.round(t7_loss_gap, 3).tolist())  # -> [0.0, 0.0, 0.8]
assert t7_noisy_loss[-1] > t7_clean_loss[-1]

plt.figure(figsize=(4.8, 2.8))
plt.bar(np.arange(3) - 0.18, t7_clean_loss, width=0.36, label="clean", color="gray")
plt.bar(np.arange(3) + 0.18, t7_noisy_loss, width=0.36, label="flipped pair", color="#e15759")
plt.xlabel("pair")
plt.ylabel("loss")
plt.title("Toy 7 · label noise")
plt.legend()
plt.show()

▶ What you'll see: only the flipped third pair jumps in loss, pushing the update in the wrong direction.


## 🛠️ Setup

In [ ]:
import numpy as np # load NumPy for arrays, exponentials, logs, and tiny optimization loops.
import matplotlib.pyplot as plt # load Matplotlib for visualizing losses, margins, and toy training curves.
np.random.seed(0) # make all stochastic examples reproducible across notebook runs.

## 🟢 Basics (warm-up)

### Basic 1 — Store one preference pair

**Goal.** Put one prompt's chosen and rejected log-probabilities into arrays, because preference optimization starts from pairwise comparisons rather than single labels. We build it in 2 steps.

In [ ]:
logp_policy_b1 = np.array([-1.0, -2.0]) # Store policy log-probabilities as [chosen, rejected].
logp_ref_b1 = np.array([-1.5, -2.2]) # Store frozen-reference log-probabilities in the same order.
print("policy [chosen, rejected]:", logp_policy_b1) # Inspect the trainable model scores.
print("reference [chosen, rejected]:", logp_ref_b1) # Inspect the anchor model scores.

▶ What you'll see: two aligned arrays containing the chosen and rejected response log-probabilities.

In [ ]:
plt.figure(figsize=(4, 3)) # Create a compact comparison plot.
plt.bar(["πθ chosen", "πθ rejected", "π0 chosen", "π0 rejected"], np.r_[logp_policy_b1, logp_ref_b1], color=["teal", "orange", "gray", "lightgray"]) # Show all four log-probabilities.
plt.axhline(0, color="black", linewidth=0.8) # Mark zero so negative log-probabilities are visually clear.
plt.title("Basic 1: one preference pair") # Title the plot.
plt.ylabel("log probability") # Label the vertical scale.
plt.xticks(rotation=20) # Rotate labels so they fit.
plt.show() # Display the chart.

▶ What you'll see: chosen is less negative than rejected under both policy and reference.

👀 Takeaway: DPO and IPO train on paired log-probability comparisons, not isolated good/bad scores.

### Basic 2 — Compute policy and reference margins

**Goal.** Compute chosen-minus-rejected margins for the policy and reference, because DPO cares about the relative odds assigned by each model. We build it in 2 steps.

In [ ]:
logp_policy_b2 = np.array([-1.0, -2.0]) # Define [chosen, rejected] policy log-probabilities.
logp_ref_b2 = np.array([-1.5, -2.2]) # Define [chosen, rejected] reference log-probabilities.
policy_margin_b2 = logp_policy_b2[0] - logp_policy_b2[1] # Compute log pi_theta chosen minus rejected.
ref_margin_b2 = logp_ref_b2[0] - logp_ref_b2[1] # Compute log pi_0 chosen minus rejected.
print("policy margin:", round(policy_margin_b2, 3)) # Inspect the policy preference margin.
print("reference margin:", round(ref_margin_b2, 3)) # Inspect the reference preference margin.

▶ What you'll see: the policy margin is 1.0 while the reference margin is 0.7.

In [ ]:
dpo_gap_b2 = policy_margin_b2 - ref_margin_b2 # Subtract the reference margin to get the DPO gap.
print("DPO gap:", round(dpo_gap_b2, 3)) # Inspect the reference-adjusted preference gap.
assert round(dpo_gap_b2, 3) == 0.300 # Verify the lesson number from the content block.
plt.figure(figsize=(4, 3)) # Create a margin comparison plot.
plt.bar(["policy margin", "reference margin", "DPO gap"], [policy_margin_b2, ref_margin_b2, dpo_gap_b2], color=["teal", "gray", "purple"]) # Show the subtraction pieces.
plt.title("Basic 2: margins become a gap") # Title the diagnostic plot.
plt.ylabel("log-odds margin") # Label the margin scale.
plt.show() # Display the chart.

▶ What you'll see: DPO keeps only the policy's improvement over the reference margin.

👀 Takeaway: the DPO gap is a reference-adjusted chosen-over-rejected log-odds improvement.

### Basic 3 — Turn a DPO gap into a loss

**Goal.** Apply the DPO loss $-\log\sigma(\beta g)$, because a preference pair becomes a binary-classification objective. We build it in 3 steps.

In [ ]:
gap_b3 = 0.3 # Use the worked DPO gap from the content block.
beta_b3 = 2.0 # Choose the worked beta value.
logit_b3 = beta_b3 * gap_b3 # Scale the gap before the sigmoid.
print("logit beta*g:", round(logit_b3, 3)) # Inspect the classification logit.

▶ What you'll see: beta=2 turns the 0.3 gap into a 0.6 logit.

In [ ]:
prob_b3 = 1 / (1 + np.exp(-logit_b3)) # Compute sigmoid(logit) from scratch.
loss_b3 = -np.log(prob_b3) # Compute the positive-label cross-entropy.
print("prob chosen wins:", round(prob_b3, 3)) # Inspect the implied preference probability.
print("DPO loss:", round(loss_b3, 3)) # Inspect the loss.
assert round(prob_b3, 3) == 0.646 and round(loss_b3, 3) == 0.437 # Verify the content-block numbers.

In [ ]:
x_b3 = np.linspace(-2, 2, 80) # Create possible DPO gaps for a loss curve.
y_b3 = -np.log(1 / (1 + np.exp(-beta_b3 * x_b3))) # Compute DPO loss for each gap.
plt.figure(figsize=(4, 3)) # Create a compact loss plot.
plt.plot(x_b3, y_b3, color="purple") # Draw the loss curve.
plt.scatter([gap_b3], [loss_b3], color="red") # Mark the worked example.
plt.title("Basic 3: DPO loss curve") # Title the plot.
plt.xlabel("gap") # Label the gap axis.
plt.ylabel("loss") # Label the loss axis.
plt.show() # Display the curve.

▶ What you'll see: positive gaps lower the loss, while negative gaps are punished strongly.

👀 Takeaway: DPO is logistic regression where the feature is a reference-adjusted preference gap.

### Basic 4 — Compare beta values

**Goal.** Sweep beta on the same gap, because beta controls how hard the preference label pushes the policy. We build it in 2 steps.

In [ ]:
gap_b4 = 0.3 # Hold the same preference gap fixed.
betas_b4 = np.array([0.5, 1.0, 2.0, 4.0]) # Try several preference-pressure settings.
probs_b4 = 1 / (1 + np.exp(-betas_b4 * gap_b4)) # Compute sigmoid(beta*gap) for each beta.
print("betas:", betas_b4) # Inspect the sweep.
print("probabilities:", np.round(probs_b4, 3)) # Inspect how beta changes confidence.
assert round(float(probs_b4[1]), 3) == 0.574 and round(float(probs_b4[2]), 3) == 0.646 # Verify worked beta values.

▶ What you'll see: the same 0.3 gap looks more decisive as beta grows.

In [ ]:
plt.figure(figsize=(4, 3)) # Create a beta sweep plot.
plt.plot(betas_b4, probs_b4, marker="o", color="teal") # Draw preference probability versus beta.
plt.title("Basic 4: beta pressure") # Title the plot.
plt.xlabel("beta") # Label beta axis.
plt.ylabel("sigmoid(beta gap)") # Label probability axis.
plt.ylim(0.5, 0.8) # Zoom into the relevant range.
plt.show() # Display the sweep.

▶ What you'll see: beta raises the implied win probability without changing the underlying pair.

👀 Takeaway: beta is a calibration knob for preference strength and KL pressure.

### Basic 5 — Compute implicit rewards

**Goal.** Convert policy/reference log-ratios into implicit rewards, because DPO hides a reward model inside the log-probability ratio. We build it in 2 steps.

In [ ]:
logp_policy_b5 = np.array([-1.0, -2.0]) # Store [chosen, rejected] policy log-probabilities.
logp_ref_b5 = np.array([-1.5, -2.2]) # Store [chosen, rejected] reference log-probabilities.
beta_b5 = 2.0 # Use the worked beta.
rewards_b5 = beta_b5 * (logp_policy_b5 - logp_ref_b5) # Compute beta times policy/reference log-ratio.
print("implicit rewards [chosen, rejected]:", np.round(rewards_b5, 3)) # Inspect rewards.

▶ What you'll see: chosen receives reward 1.0 and rejected receives reward 0.4.

In [ ]:
reward_gap_b5 = rewards_b5[0] - rewards_b5[1] # Difference the implicit rewards.
print("reward gap:", round(reward_gap_b5, 3)) # Inspect the DPO logit.
assert round(reward_gap_b5, 3) == 0.600 # Verify that reward gap equals beta times DPO gap.
plt.figure(figsize=(4, 3)) # Create an implicit-reward bar chart.
plt.bar(["chosen", "rejected"], rewards_b5, color=["teal", "orange"]) # Plot the two rewards.
plt.title("Basic 5: implicit reward") # Title the plot.
plt.ylabel("beta · (logπθ - logπ0)") # Label reward scale.
plt.show() # Display the chart.

▶ What you'll see: DPO's classifier logit is the chosen reward minus rejected reward.

👀 Takeaway: DPO avoids an explicit reward model but still optimizes an implicit policy/reference reward.

### Basic 6 — See why the reference term matters

**Goal.** Compare DPO's reference-adjusted gap with a raw policy gap, because dropping the reference loses the KL anchor. We build it in 2 steps.

In [ ]:
policy_margin_b6 = 1.0 # Chosen minus rejected under the trainable policy.
reference_margin_b6 = 0.7 # Chosen minus rejected under the frozen reference.
raw_gap_b6 = policy_margin_b6 # Raw likelihood preference would keep the full policy margin.
anchored_gap_b6 = policy_margin_b6 - reference_margin_b6 # DPO keeps only improvement over reference.
print("raw policy gap:", raw_gap_b6) # Inspect the unanchored objective signal.
print("reference-adjusted gap:", round(anchored_gap_b6, 3)) # Inspect the DPO signal.

▶ What you'll see: the raw gap is much larger because it includes preference the reference already had.

In [ ]:
plt.figure(figsize=(4, 3)) # Create a comparison plot.
plt.bar(["raw policy", "DPO anchored"], [raw_gap_b6, anchored_gap_b6], color=["crimson", "seagreen"]) # Compare the two signals.
plt.title("Basic 6: reference anchor") # Title the plot.
plt.ylabel("gap used by loss") # Label the vertical axis.
plt.show() # Display the chart.

▶ What you'll see: DPO's signal is smaller because it asks what changed relative to the base model.

👀 Takeaway: reference terms keep preference tuning from becoming plain chosen-likelihood maximization.

### Basic 7 — Compute the IPO target

**Goal.** Compute IPO's finite target gap, because IPO treats the preference margin as a regression target rather than an always-grow margin. We build it in 2 steps.

In [ ]:
beta_b7 = 2.0 # Choose the worked beta value.
actual_gap_b7 = 0.3 # Use the worked policy-reference gap.
target_gap_b7 = 1 / (2 * beta_b7) # Compute the simplified IPO target gap.
print("target gap:", round(target_gap_b7, 3)) # Inspect the desired finite gap.
print("actual gap:", round(actual_gap_b7, 3)) # Inspect the current gap.
assert round(target_gap_b7, 3) == 0.250 # Verify the content-block IPO target.

▶ What you'll see: IPO wants a 0.25 gap when beta is 2.

In [ ]:
ipo_loss_b7 = (actual_gap_b7 - target_gap_b7) ** 2 # Compute squared error to the target.
print("IPO loss:", round(ipo_loss_b7, 4)) # Inspect the small regression loss.
assert round(ipo_loss_b7, 4) == 0.0025 # Verify the content-block IPO loss.
plt.figure(figsize=(4, 3)) # Create a simple target plot.
plt.bar(["actual", "target"], [actual_gap_b7, target_gap_b7], color=["orange", "gray"]) # Compare actual and target gaps.
plt.title("Basic 7: IPO target gap") # Title the plot.
plt.ylabel("gap") # Label the gap axis.
plt.show() # Display the chart.

▶ What you'll see: the actual gap is slightly above target, so the squared error is small.

👀 Takeaway: IPO makes the desired amount of preference separation explicit and finite.

### Basic 8 — Plot DPO versus IPO on one gap axis

**Goal.** Compare the shapes of DPO and IPO, because their losses encourage different margin behavior. We build it in 2 steps.

In [ ]:
gaps_b8 = np.linspace(-0.5, 1.2, 120) # Create possible policy-reference gaps.
beta_b8 = 2.0 # Use the same beta for both losses.
dpo_b8 = -np.log(1 / (1 + np.exp(-beta_b8 * gaps_b8))) # Compute DPO logistic loss.
ipo_b8 = (gaps_b8 - 1 / (2 * beta_b8)) ** 2 # Compute IPO squared error to target.
print("DPO loss at gap .3:", round(float(-np.log(1 / (1 + np.exp(-beta_b8 * 0.3)))), 3)) # Inspect the worked DPO point.
print("IPO loss at gap .3:", round(float((0.3 - 1 / (2 * beta_b8)) ** 2), 4)) # Inspect the worked IPO point.

▶ What you'll see: the two objectives assign very different numeric penalties to the same gap.

In [ ]:
plt.figure(figsize=(4.6, 3)) # Create a loss comparison plot.
plt.plot(gaps_b8, dpo_b8, label="DPO", color="purple") # Draw DPO loss.
plt.plot(gaps_b8, ipo_b8, label="IPO", color="darkorange") # Draw IPO loss.
plt.axvline(1 / (2 * beta_b8), color="gray", linestyle="--", label="IPO target") # Mark IPO target.
plt.title("Basic 8: DPO vs IPO shapes") # Title the plot.
plt.xlabel("gap") # Label gap axis.
plt.ylabel("loss") # Label loss axis.
plt.legend() # Show curve labels.
plt.show() # Display the comparison.

▶ What you'll see: DPO keeps decreasing as the gap grows, while IPO bottoms out at a finite gap.

👀 Takeaway: DPO is margin-increasing logistic loss; IPO is target-margin squared loss.

### Basic 9 — Average loss over a tiny batch

**Goal.** Compute DPO loss for several preference pairs at once, because training uses batches of comparisons. We build it in 2 steps.

In [ ]:
gaps_b9 = np.array([0.3, 0.1, -0.2, 0.7]) # Store four reference-adjusted preference gaps.
beta_b9 = 2.0 # Use a fixed beta across the batch.
logits_b9 = beta_b9 * gaps_b9 # Scale each gap into a classification logit.
print("logits:", np.round(logits_b9, 3)) # Inspect per-pair logits.

▶ What you'll see: one pair has a negative logit, meaning the rejected response currently wins.

In [ ]:
losses_b9 = -np.log(1 / (1 + np.exp(-logits_b9))) # Compute DPO loss per pair.
mean_loss_b9 = float(np.mean(losses_b9)) # Average the batch loss.
print("pair losses:", np.round(losses_b9, 3)) # Inspect each pair's contribution.
print("mean loss:", round(mean_loss_b9, 3)) # Inspect the training objective.
plt.figure(figsize=(4, 3)) # Create a per-pair loss chart.
plt.bar(range(len(losses_b9)), losses_b9, color="slateblue") # Show which pairs dominate.
plt.title("Basic 9: batch DPO losses") # Title the plot.
plt.xlabel("pair index") # Label pairs.
plt.ylabel("loss") # Label loss scale.
plt.show() # Display the chart.

▶ What you'll see: the negative-gap pair has the largest loss and will drive a strong update.

👀 Takeaway: batch DPO is just the mean of independent pairwise logistic losses.

### Basic 10 — Compute one gradient sign

**Goal.** Inspect the derivative with respect to the DPO gap, because gradient descent should raise gaps when chosen responses are not winning enough. We build it in 2 steps.

In [ ]:
gap_b10 = 0.3 # Use the worked gap.
beta_b10 = 2.0 # Use the worked beta.
prob_b10 = 1 / (1 + np.exp(-beta_b10 * gap_b10)) # Compute sigmoid(beta gap).
grad_gap_b10 = -beta_b10 * (1 - prob_b10) # Derivative of -log sigmoid(beta gap) with respect to gap.
print("probability:", round(prob_b10, 3)) # Inspect the current win probability.
print("dL/dgap:", round(grad_gap_b10, 3)) # Inspect the gradient sign.

▶ What you'll see: the gradient is negative, so gradient descent increases the gap.

In [ ]:
gap_after_b10 = gap_b10 - 0.1 * grad_gap_b10 # Take a tiny illustrative descent step on the gap.
loss_before_b10 = -np.log(1 / (1 + np.exp(-beta_b10 * gap_b10))) # Compute loss before the step.
loss_after_b10 = -np.log(1 / (1 + np.exp(-beta_b10 * gap_after_b10))) # Compute loss after the step.
print("gap before -> after:", round(gap_b10, 3), "->", round(gap_after_b10, 3)) # Inspect the step.
print("loss before -> after:", round(loss_before_b10, 3), "->", round(loss_after_b10, 3)) # Inspect the improvement.
assert loss_after_b10 < loss_before_b10 # Verify the step reduced DPO loss.
plt.figure(figsize=(4, 3)) # Create a before-after chart.
plt.bar(["before", "after"], [loss_before_b10, loss_after_b10], color=["gray", "teal"]) # Compare loss values.
plt.title("Basic 10: gradient raises the gap") # Title the plot.
plt.ylabel("DPO loss") # Label the loss axis.
plt.show() # Display the chart.

▶ What you'll see: the gap increases and the loss decreases after one gradient step.

👀 Takeaway: DPO gradients push chosen responses upward relative to rejected responses and the reference.

## 🟡 Easy

### Easy 1 — Implement DPO loss from arrays

**Goal.** Write the full DPO computation for a mini-batch, because real training vectorizes chosen and rejected log-probabilities. We build it in 3 steps.

In [ ]:
logp_c_e1 = np.array([-1.0, -0.8, -1.4]) # Store policy log-probabilities for chosen responses.
logp_r_e1 = np.array([-2.0, -1.3, -1.5]) # Store policy log-probabilities for rejected responses.
ref_c_e1 = np.array([-1.5, -1.0, -1.2]) # Store reference log-probabilities for chosen responses.
ref_r_e1 = np.array([-2.2, -1.4, -1.6]) # Store reference log-probabilities for rejected responses.
beta_e1 = 2.0 # Choose preference pressure.
print("batch size:", len(logp_c_e1)) # Inspect how many preference pairs are in the batch.

▶ What you'll see: three preference pairs are ready for vectorized DPO.

In [ ]:
gaps_e1 = (logp_c_e1 - ref_c_e1) - (logp_r_e1 - ref_r_e1) # Compute DPO gap for each pair.
losses_e1 = -np.log(1 / (1 + np.exp(-beta_e1 * gaps_e1))) # Compute per-pair DPO losses.
print("gaps:", np.round(gaps_e1, 3)) # Inspect the reference-adjusted margins.
print("losses:", np.round(losses_e1, 3)) # Inspect each pair's loss.
assert round(float(gaps_e1[0]), 3) == 0.300 # Verify the first pair matches the worked example.

In [ ]:
mean_e1 = float(np.mean(losses_e1)) # Average the batch loss.
print("mean DPO loss:", round(mean_e1, 3)) # Inspect the scalar objective.
plt.figure(figsize=(4, 3)) # Create a batch diagnostic chart.
plt.bar(range(len(gaps_e1)), gaps_e1, color="teal") # Plot each gap.
plt.axhline(0, color="black", linewidth=0.8) # Mark the point where chosen and rejected tie after reference adjustment.
plt.title("Easy 1: DPO gaps per pair") # Title the plot.
plt.xlabel("pair") # Label pair index.
plt.ylabel("gap") # Label gap axis.
plt.show() # Display the chart.

▶ What you'll see: positive gaps have lower losses; negative or small gaps need larger updates.

👀 Takeaway: DPO loss is a vectorized chosen/rejected log-probability margin calculation.

### Easy 2 — Train one scalar policy with DPO

**Goal.** Optimize one scalar parameter on toy preference features, because DPO is easiest to understand when the policy has just one knob. We build it in 4 steps.

In [ ]:
chosen_feat_e2 = np.array([1.0, 0.8, 1.2, 0.6]) # Define features for chosen responses.
reject_feat_e2 = np.array([0.1, 0.3, 0.2, 0.4]) # Define smaller features for rejected responses.
base_c_e2 = np.array([-1.4, -1.2, -1.8, -1.0]) # Define reference-like base chosen log-probabilities.
base_r_e2 = np.array([-1.3, -1.1, -1.6, -1.1]) # Define reference-like base rejected log-probabilities.
print("feature gaps:", chosen_feat_e2 - reject_feat_e2) # Inspect why positive theta should help.

▶ What you'll see: chosen responses have larger features than rejected responses.

In [ ]:
theta_e2 = 0.0 # Initialize the trainable scalar policy parameter.
beta_e2 = 2.0 # Choose DPO beta.
eta_e2 = 0.5 # Choose a learning rate for the toy update.
loss_hist_e2 = [] # Store loss values for plotting.
print("theta start:", theta_e2) # Inspect the initialization.

In [ ]:
for step_e2 in range(45): # Run several gradient-descent steps.
    logp_c_e2 = base_c_e2 + theta_e2 * chosen_feat_e2 # Compute chosen policy log-probabilities.
    logp_r_e2 = base_r_e2 + theta_e2 * reject_feat_e2 # Compute rejected policy log-probabilities.
    gaps_e2 = (logp_c_e2 - base_c_e2) - (logp_r_e2 - base_r_e2) # Compute reference-adjusted gaps.
    probs_e2 = 1 / (1 + np.exp(-beta_e2 * gaps_e2)) # Convert gaps to win probabilities.
    loss_hist_e2.append(float(np.mean(-np.log(probs_e2)))) # Store the mean DPO loss.
    grad_e2 = np.mean(-beta_e2 * (1 - probs_e2) * (chosen_feat_e2 - reject_feat_e2)) # Differentiate loss through the scalar theta.
    theta_e2 = theta_e2 - eta_e2 * grad_e2 # Apply gradient descent.
print("theta final:", round(theta_e2, 3)) # Inspect the learned parameter.
print("loss start/end:", round(loss_hist_e2[0], 3), round(loss_hist_e2[-1], 3)) # Inspect convergence.
assert loss_hist_e2[-1] < loss_hist_e2[0] # Verify DPO training improved the objective.

In [ ]:
plt.figure(figsize=(4.6, 3)) # Create a training curve figure.
plt.plot(loss_hist_e2, color="purple") # Plot mean DPO loss by step.
plt.title("Easy 2: one-parameter DPO training") # Title the curve.
plt.xlabel("step") # Label training steps.
plt.ylabel("mean DPO loss") # Label objective.
plt.show() # Display the curve.

▶ What you'll see: loss decreases as theta learns to favor chosen-response features.

👀 Takeaway: DPO is ordinary gradient descent once the pairwise log-probability gap is computed.

### Easy 3 — Compare DPO and IPO updates

**Goal.** Train the same scalar toy model with DPO and IPO, because DPO keeps increasing margins while IPO aims for a target margin. We build it in 4 steps.

In [ ]:
feat_c_e3 = np.array([1.0, 0.8, 1.2, 0.6]) # Define chosen features.
feat_r_e3 = np.array([0.1, 0.3, 0.2, 0.4]) # Define rejected features.
delta_feat_e3 = feat_c_e3 - feat_r_e3 # Compute the feature effect on the preference gap.
beta_e3 = 2.0 # Use the same beta for both objectives.
target_e3 = 1 / (2 * beta_e3) # Compute IPO target gap.
print("IPO target:", round(target_e3, 3)) # Inspect the finite target margin.

▶ What you'll see: IPO wants the average pair gap to sit around 0.25.

In [ ]:
theta_dpo_e3 = 0.0 # Initialize DPO parameter.
theta_ipo_e3 = 0.0 # Initialize IPO parameter.
hist_dpo_e3 = [] # Store average DPO gaps.
hist_ipo_e3 = [] # Store average IPO gaps.
print("both methods start at zero gap") # Confirm matching initialization.

In [ ]:
for step_e3 in range(60): # Train both objectives side by side.
    gaps_dpo_e3 = theta_dpo_e3 * delta_feat_e3 # DPO gaps from scalar parameter.
    probs_e3 = 1 / (1 + np.exp(-beta_e3 * gaps_dpo_e3)) # DPO probabilities.
    grad_dpo_e3 = np.mean(-beta_e3 * (1 - probs_e3) * delta_feat_e3) # DPO gradient.
    theta_dpo_e3 -= 0.4 * grad_dpo_e3 # Apply DPO update.
    gaps_ipo_e3 = theta_ipo_e3 * delta_feat_e3 # IPO gaps from scalar parameter.
    grad_ipo_e3 = np.mean(2 * (gaps_ipo_e3 - target_e3) * delta_feat_e3) # IPO squared-error gradient.
    theta_ipo_e3 -= 0.4 * grad_ipo_e3 # Apply IPO update.
    hist_dpo_e3.append(float(np.mean(gaps_dpo_e3))) # Store average DPO gap.
    hist_ipo_e3.append(float(np.mean(gaps_ipo_e3))) # Store average IPO gap.
print("final average gaps DPO/IPO:", round(hist_dpo_e3[-1], 3), round(hist_ipo_e3[-1], 3)) # Inspect final margins.

In [ ]:
plt.figure(figsize=(4.6, 3)) # Create a gap trajectory plot.
plt.plot(hist_dpo_e3, label="DPO avg gap", color="purple") # Plot DPO margin growth.
plt.plot(hist_ipo_e3, label="IPO avg gap", color="orange") # Plot IPO margin approach.
plt.axhline(target_e3, color="gray", linestyle="--", label="IPO target") # Mark finite IPO target.
plt.title("Easy 3: DPO vs IPO gap behavior") # Title the plot.
plt.xlabel("step") # Label steps.
plt.ylabel("average gap") # Label average gap.
plt.legend() # Show curve labels.
plt.show() # Display the comparison.

▶ What you'll see: DPO's gap keeps growing, while IPO levels off near its target.

👀 Takeaway: DPO encourages larger winning margins; IPO regularizes toward a chosen finite margin.

### Easy 4 — Visualize beta saturation

**Goal.** Show how large beta can reduce useful gradients, because sigmoid saturation weakens updates on already-separated pairs. We build it in 3 steps.

In [ ]:
gaps_e4 = np.linspace(-2, 2, 120) # Create a range of DPO gaps.
betas_e4 = np.array([0.5, 2.0, 8.0]) # Compare mild, standard, and large beta values.
print("beta values:", betas_e4) # Inspect the sweep.

▶ What you'll see: three beta settings will be compared on the same gaps.

In [ ]:
grad_curves_e4 = [] # Store gradient magnitudes for each beta.
for beta_e4 in betas_e4: # Loop over beta values.
    probs_e4 = 1 / (1 + np.exp(-beta_e4 * gaps_e4)) # Compute sigmoid(beta gap).
    grad_curves_e4.append(beta_e4 * (1 - probs_e4)) # Store magnitude of the positive-label gradient.
print("gradient at gap=1 for beta=8:", round(float(8 * (1 - 1 / (1 + np.exp(-8)))), 3)) # Inspect saturation on a correct pair.

In [ ]:
plt.figure(figsize=(4.8, 3)) # Create a gradient plot.
for beta_e4, grad_e4 in zip(betas_e4, grad_curves_e4): # Plot each beta curve.
    plt.plot(gaps_e4, grad_e4, label=f"beta={beta_e4}") # Draw gradient magnitude against gap.
plt.title("Easy 4: sigmoid gradient saturation") # Title the plot.
plt.xlabel("gap") # Label the gap axis.
plt.ylabel("|dL/dgap|") # Label gradient magnitude.
plt.legend() # Show beta labels.
plt.show() # Display the chart.

▶ What you'll see: large beta creates very steep gradients near wrong pairs but tiny gradients on already-correct positive gaps.

👀 Takeaway: beta trades off preference force against saturation and should not be treated as harmless scaling.

### Easy 5 — Detect noisy preference labels

**Goal.** Compare clean and flipped preference labels, because DPO cannot fix pairwise data quality by itself. We build it in 3 steps.

In [ ]:
gaps_e5 = np.array([1.0, 0.8, 0.5, 0.2]) # Define model gaps that correctly favor the chosen response.
labels_clean_e5 = np.ones(4) # Clean labels say chosen should win for every pair.
labels_noisy_e5 = np.array([1, 1, -1, 1]) # Flip the third label so rejected is incorrectly treated as chosen.
print("signed clean gaps:", labels_clean_e5 * gaps_e5) # Inspect clean training signs.
print("signed noisy gaps:", labels_noisy_e5 * gaps_e5) # Inspect noisy training signs.

▶ What you'll see: the flipped label turns a good positive gap into a negative training example.

In [ ]:
loss_clean_e5 = -np.log(1 / (1 + np.exp(-(labels_clean_e5 * gaps_e5)))) # Compute clean pair losses.
loss_noisy_e5 = -np.log(1 / (1 + np.exp(-(labels_noisy_e5 * gaps_e5)))) # Compute noisy pair losses.
print("clean losses:", np.round(loss_clean_e5, 3)) # Inspect clean objective contributions.
print("noisy losses:", np.round(loss_noisy_e5, 3)) # Inspect noisy objective contributions.

In [ ]:
plt.figure(figsize=(4.6, 3)) # Create a grouped loss chart.
idx_e5 = np.arange(len(gaps_e5)) # Create bar positions.
plt.bar(idx_e5 - 0.18, loss_clean_e5, width=0.36, label="clean", color="teal") # Plot clean losses.
plt.bar(idx_e5 + 0.18, loss_noisy_e5, width=0.36, label="one flip", color="crimson") # Plot noisy losses.
plt.title("Easy 5: label noise changes DPO") # Title the plot.
plt.xlabel("pair") # Label pair index.
plt.ylabel("loss") # Label loss scale.
plt.legend() # Show legend.
plt.show() # Display the chart.

▶ What you'll see: the flipped pair receives a large loss and would push the model in the wrong direction.

👀 Takeaway: direct preference optimization removes the RL loop, not the need for reliable preference data.

## 🔴 Advanced

### Advanced 1 — Sweep beta with train and validation losses

**Goal.** Train a scalar DPO model for several beta values and evaluate on held-out pairs, because beta should be chosen with validation rather than training loss alone. We build it in 5 steps.

In [ ]:
feat_c_a1 = np.array([1.0, 0.9, 1.1, 0.7, 1.2, 0.8]) # Define chosen features for six pairs.
feat_r_a1 = np.array([0.2, 0.4, 0.1, 0.3, 0.5, 0.2]) # Define rejected features for six pairs.
train_idx_a1 = np.array([0, 1, 2, 3]) # Use four pairs for training.
val_idx_a1 = np.array([4, 5]) # Hold out two pairs for validation.
print("train pairs:", train_idx_a1, "validation pairs:", val_idx_a1) # Inspect the split.

▶ What you'll see: a tiny train/validation setup for beta selection.

In [ ]:
betas_a1 = np.array([0.5, 1.0, 2.0, 5.0]) # Candidate beta values.
train_losses_a1 = [] # Store final training losses.
val_losses_a1 = [] # Store validation losses.
thetas_a1 = [] # Store learned parameters.
print("beta grid:", betas_a1) # Inspect hyperparameters.

In [ ]:
for beta_a1 in betas_a1: # Train one scalar model per beta.
    theta_a1 = 0.0 # Reset initialization for fair comparison.
    delta_train_a1 = feat_c_a1[train_idx_a1] - feat_r_a1[train_idx_a1] # Training feature gaps.
    for step_a1 in range(80): # Run gradient descent.
        gaps_train_a1 = theta_a1 * delta_train_a1 # Compute train gaps.
        probs_train_a1 = 1 / (1 + np.exp(-beta_a1 * gaps_train_a1)) # Compute DPO probabilities.
        grad_a1 = np.mean(-beta_a1 * (1 - probs_train_a1) * delta_train_a1) # Differentiate mean train loss.
        theta_a1 -= 0.3 * grad_a1 # Apply update.
    delta_all_a1 = feat_c_a1 - feat_r_a1 # Compute all feature gaps.
    gaps_all_a1 = theta_a1 * delta_all_a1 # Score all pairs.
    losses_all_a1 = -np.log(1 / (1 + np.exp(-beta_a1 * gaps_all_a1))) # Compute all DPO losses.
    train_losses_a1.append(float(np.mean(losses_all_a1[train_idx_a1]))) # Save train loss.
    val_losses_a1.append(float(np.mean(losses_all_a1[val_idx_a1]))) # Save validation loss.
    thetas_a1.append(theta_a1) # Save learned theta.
print("thetas:", np.round(thetas_a1, 3)) # Inspect learned parameters.
print("validation losses:", np.round(val_losses_a1, 3)) # Inspect held-out losses.

In [ ]:
best_i_a1 = int(np.argmin(val_losses_a1)) # Select beta by validation loss.
best_beta_a1 = float(betas_a1[best_i_a1]) # Read the winning beta.
print("best beta:", best_beta_a1) # Inspect selection.
assert min(val_losses_a1) == val_losses_a1[best_i_a1] # Verify the selected index is consistent.

In [ ]:
plt.figure(figsize=(4.8, 3)) # Create a beta-selection plot.
plt.plot(betas_a1, train_losses_a1, marker="o", label="train") # Plot training loss.
plt.plot(betas_a1, val_losses_a1, marker="s", label="validation") # Plot validation loss.
plt.axvline(best_beta_a1, color="red", linestyle="--", label="best beta") # Mark selected beta.
plt.title("Advanced 1: beta validation sweep") # Title the plot.
plt.xlabel("beta") # Label beta axis.
plt.ylabel("DPO loss") # Label loss axis.
plt.legend() # Show labels.
plt.show() # Display the chart.

▶ What you'll see: training and validation curves may prefer different amounts of preference pressure.

👀 Takeaway: beta is a real hyperparameter, so tune it on held-out preference data.

### Advanced 2 — Compare anchored and unanchored training

**Goal.** Train with and without reference subtraction, because the unanchored objective can move probability mass that the reference already assigned correctly. We build it in 5 steps.

In [ ]:
ref_margin_a2 = np.array([1.5, 1.2, 1.0, 0.8]) # Reference already prefers chosen responses by these margins.
delta_feat_a2 = np.array([0.7, 0.6, 0.8, 0.5]) # Trainable parameter's effect on chosen-vs-rejected margin.
beta_a2 = 2.0 # Use a fixed beta.
print("reference margins:", ref_margin_a2) # Inspect the base model preference.

▶ What you'll see: the reference model already has strong chosen-over-rejected margins.

In [ ]:
theta_anchor_a2 = 0.0 # Initialize reference-anchored DPO parameter.
theta_raw_a2 = 0.0 # Initialize unanchored parameter.
hist_anchor_a2 = [] # Store anchored theta values.
hist_raw_a2 = [] # Store raw theta values.
print("initial thetas:", theta_anchor_a2, theta_raw_a2) # Inspect matching starts.

In [ ]:
for step_a2 in range(50): # Train both objectives side by side.
    gap_anchor_a2 = theta_anchor_a2 * delta_feat_a2 # Anchored DPO uses only improvement over reference.
    prob_anchor_a2 = 1 / (1 + np.exp(-beta_a2 * gap_anchor_a2)) # Compute anchored probabilities.
    grad_anchor_a2 = np.mean(-beta_a2 * (1 - prob_anchor_a2) * delta_feat_a2) # Anchored gradient.
    theta_anchor_a2 -= 0.25 * grad_anchor_a2 # Update anchored model.
    gap_raw_a2 = ref_margin_a2 + theta_raw_a2 * delta_feat_a2 # Raw objective includes reference preference again.
    prob_raw_a2 = 1 / (1 + np.exp(-beta_a2 * gap_raw_a2)) # Compute raw probabilities.
    grad_raw_a2 = np.mean(-beta_a2 * (1 - prob_raw_a2) * delta_feat_a2) # Raw gradient.
    theta_raw_a2 -= 0.25 * grad_raw_a2 # Update raw model.
    hist_anchor_a2.append(theta_anchor_a2) # Store anchored trajectory.
    hist_raw_a2.append(theta_raw_a2) # Store raw trajectory.
print("final thetas anchored/raw:", round(theta_anchor_a2, 3), round(theta_raw_a2, 3)) # Inspect how objectives differ.

In [ ]:
final_gap_anchor_a2 = theta_anchor_a2 * delta_feat_a2 # Compute final anchored improvements.
final_gap_raw_a2 = ref_margin_a2 + theta_raw_a2 * delta_feat_a2 # Compute final raw margins.
print("mean anchored improvement:", round(float(np.mean(final_gap_anchor_a2)), 3)) # Inspect improvement over reference.
print("mean raw margin:", round(float(np.mean(final_gap_raw_a2)), 3)) # Inspect total margin.

In [ ]:
plt.figure(figsize=(4.8, 3)) # Create a trajectory plot.
plt.plot(hist_anchor_a2, label="anchored θ", color="seagreen") # Plot anchored parameter.
plt.plot(hist_raw_a2, label="raw θ", color="crimson") # Plot raw parameter.
plt.title("Advanced 2: reference subtraction changes training") # Title the plot.
plt.xlabel("step") # Label steps.
plt.ylabel("theta") # Label parameter value.
plt.legend() # Show labels.
plt.show() # Display the chart.

▶ What you'll see: the unanchored objective sees easier already-positive margins and produces a different update path.

👀 Takeaway: the reference term changes the optimization target, not just the reporting metric.

### Advanced 3 — Robustness to label noise

**Goal.** Sweep label-flip rates and measure validation accuracy, because preference optimization follows the labels it is given. We build it in 4 steps.

In [ ]:
rng_a3 = np.random.default_rng(3) # Create reproducible synthetic preference data.
delta_feat_a3 = rng_a3.uniform(0.2, 1.2, size=40) # Positive feature gaps mean chosen should usually win.
true_labels_a3 = np.ones(40) # Clean preference labels.
flip_rates_a3 = np.array([0.0, 0.2, 0.5, 0.8]) # Test increasing label noise, including a majority-flipped case.
print("flip rates:", flip_rates_a3) # Inspect noise settings.

▶ What you'll see: a synthetic preference task with controllable label flips.

In [ ]:
train_acc_a3 = [] # Store training signed-margin accuracy.
val_acc_a3 = [] # Store validation signed-margin accuracy against clean labels.
learned_theta_a3 = [] # Store learned scalar parameters.
print("train size 30, validation size 10") # Document the split.

In [ ]:
for flip_a3 in flip_rates_a3: # Train once per noise level.
    labels_a3 = true_labels_a3.copy() # Start from clean labels.
    n_flip_a3 = int(flip_a3 * 30) # Flip only training labels.
    labels_a3[:n_flip_a3] = -1 # Turn chosen/rejected direction around for noisy pairs.
    theta_a3 = 0.0 # Initialize scalar model.
    for step_a3 in range(80): # Run DPO-like signed logistic training.
        signed_gap_a3 = labels_a3[:30] * theta_a3 * delta_feat_a3[:30] # Use noisy signs in training.
        probs_a3 = 1 / (1 + np.exp(-signed_gap_a3)) # Compute signed win probabilities.
        grad_a3 = np.mean(-(1 - probs_a3) * labels_a3[:30] * delta_feat_a3[:30]) # Differentiate signed logistic loss.
        theta_a3 -= 0.4 * grad_a3 # Apply update.
    learned_theta_a3.append(theta_a3) # Save theta.
    train_acc_a3.append(float(np.mean(labels_a3[:30] * theta_a3 * delta_feat_a3[:30] > 0))) # Accuracy on noisy training signs.
    val_acc_a3.append(float(np.mean(true_labels_a3[30:] * theta_a3 * delta_feat_a3[30:] > 0))) # Accuracy on clean validation signs.
print("learned theta:", np.round(learned_theta_a3, 3)) # Inspect learned direction.
print("validation accuracy:", np.round(val_acc_a3, 3)) # Inspect clean held-out quality.

In [ ]:
plt.figure(figsize=(4.8, 3)) # Create a noise robustness plot.
plt.plot(flip_rates_a3, train_acc_a3, marker="o", label="train noisy-label acc") # Plot noisy training accuracy.
plt.plot(flip_rates_a3, val_acc_a3, marker="s", label="clean validation acc") # Plot clean validation accuracy.
plt.title("Advanced 3: label noise hurts preference tuning") # Title the plot.
plt.xlabel("training label flip rate") # Label noise axis.
plt.ylabel("accuracy") # Label accuracy axis.
plt.ylim(-0.05, 1.05) # Keep accuracy scale readable.
plt.legend() # Show labels.
plt.show() # Display the chart.

▶ What you'll see: enough flipped preferences can reduce or destroy clean validation performance.

👀 Takeaway: DPO and IPO need preference-quality controls just as much as reward-model pipelines do.

### Advanced 4 — Token-level sequence log-probabilities

**Goal.** Sum token log-probabilities into response log-probabilities, because DPO compares entire chosen and rejected completions. We build it in 4 steps.

In [ ]:
token_logp_chosen_a4 = np.array([-0.2, -0.4, -0.3, -0.5]) # Policy token log-probs for a chosen response.
token_logp_reject_a4 = np.array([-0.3, -0.8, -0.7]) # Policy token log-probs for a rejected response.
token_ref_chosen_a4 = np.array([-0.25, -0.45, -0.35, -0.55]) # Reference token log-probs for chosen response.
token_ref_reject_a4 = np.array([-0.3, -0.8, -0.8]) # Reference token log-probs for rejected response.
print("chosen tokens:", len(token_logp_chosen_a4), "rejected tokens:", len(token_logp_reject_a4)) # Inspect sequence lengths.

▶ What you'll see: the responses have different lengths, but each has a total log-probability.

In [ ]:
seq_logp_c_a4 = float(np.sum(token_logp_chosen_a4)) # Sum chosen token logs into sequence log-probability.
seq_logp_r_a4 = float(np.sum(token_logp_reject_a4)) # Sum rejected token logs into sequence log-probability.
seq_ref_c_a4 = float(np.sum(token_ref_chosen_a4)) # Sum reference chosen token logs.
seq_ref_r_a4 = float(np.sum(token_ref_reject_a4)) # Sum reference rejected token logs.
print("policy sequence logs:", round(seq_logp_c_a4, 3), round(seq_logp_r_a4, 3)) # Inspect sequence scores.
print("reference sequence logs:", round(seq_ref_c_a4, 3), round(seq_ref_r_a4, 3)) # Inspect reference sequence scores.

In [ ]:
gap_a4 = (seq_logp_c_a4 - seq_ref_c_a4) - (seq_logp_r_a4 - seq_ref_r_a4) # Compute DPO gap from sequence sums.
loss_a4 = -np.log(1 / (1 + np.exp(-2.0 * gap_a4))) # Compute DPO loss with beta=2.
print("sequence DPO gap:", round(gap_a4, 3)) # Inspect the pairwise gap.
print("sequence DPO loss:", round(loss_a4, 3)) # Inspect loss.
assert round(gap_a4, 3) == 0.100 # Verify the hand-built sequence gap.

In [ ]:
plt.figure(figsize=(4.8, 3)) # Create a token contribution plot.
plt.bar(np.arange(len(token_logp_chosen_a4)), token_logp_chosen_a4 - token_ref_chosen_a4, color="teal", label="chosen token ratio") # Plot chosen token log-ratio contributions.
plt.bar(np.arange(len(token_logp_reject_a4)) + 0.15, token_logp_reject_a4 - token_ref_reject_a4, width=0.35, color="orange", label="rejected token ratio") # Plot rejected token log-ratio contributions.
plt.axhline(0, color="black", linewidth=0.8) # Mark no policy/reference change.
plt.title("Advanced 4: token sums make sequence gaps") # Title the plot.
plt.xlabel("token position") # Label token positions.
plt.ylabel("logπθ - logπ0") # Label contribution scale.
plt.legend() # Show labels.
plt.show() # Display the chart.

▶ What you'll see: sequence-level preference gaps are sums of token-level policy/reference log-ratio contributions.

👀 Takeaway: DPO operates on complete response log-probabilities, which are token log-probability sums.

### Advanced 5 — Calibrate implicit reward scale

**Goal.** Compare implicit reward distributions for several beta values, because beta rescales policy/reference ratios and changes how separated responses appear. We build it in 4 steps.

In [ ]:
log_ratio_chosen_a5 = np.array([0.5, 0.2, 0.7, 0.1, -0.1]) # Chosen logπθ-logπ0 values.
log_ratio_reject_a5 = np.array([0.2, 0.0, 0.1, 0.3, -0.2]) # Rejected logπθ-logπ0 values.
betas_a5 = np.array([0.5, 1.0, 2.0, 4.0]) # Reward scale candidates.
raw_gaps_a5 = log_ratio_chosen_a5 - log_ratio_reject_a5 # Compute unscaled preference gaps.
print("raw gaps:", np.round(raw_gaps_a5, 3)) # Inspect pairwise policy/reference improvements.

▶ What you'll see: most chosen responses improved more than rejected responses, but one pair is negative.

In [ ]:
mean_reward_gap_a5 = [] # Store average beta-scaled reward gap.
win_prob_a5 = [] # Store average DPO win probability.
for beta_a5 in betas_a5: # Sweep beta values.
    reward_gap_a5 = beta_a5 * raw_gaps_a5 # Scale implicit reward gaps.
    mean_reward_gap_a5.append(float(np.mean(reward_gap_a5))) # Save mean reward separation.
    win_prob_a5.append(float(np.mean(1 / (1 + np.exp(-reward_gap_a5))))) # Save mean pair win probability.
print("mean reward gaps:", np.round(mean_reward_gap_a5, 3)) # Inspect reward scaling.
print("mean win probs:", np.round(win_prob_a5, 3)) # Inspect probability scaling.

In [ ]:
assert round(float(raw_gaps_a5[0]), 3) == 0.300 # Verify the first pair uses the content-block gap.
print("first pair beta=2 reward gap:", round(float(2.0 * raw_gaps_a5[0]), 3)) # Inspect beta-scaled first gap.
assert round(float(2.0 * raw_gaps_a5[0]), 3) == 0.600 # Verify the worked logit.

In [ ]:
plt.figure(figsize=(4.8, 3)) # Create a scale calibration plot.
plt.plot(betas_a5, mean_reward_gap_a5, marker="o", label="mean reward gap") # Plot reward gap scale.
plt.plot(betas_a5, win_prob_a5, marker="s", label="mean win probability") # Plot implied probabilities.
plt.title("Advanced 5: beta scales implicit rewards") # Title the plot.
plt.xlabel("beta") # Label beta axis.
plt.legend() # Show labels.
plt.show() # Display the chart.

▶ What you'll see: beta linearly scales reward gaps but nonlinearly changes win probabilities through the sigmoid.

👀 Takeaway: implicit rewards are policy/reference log-ratios, and beta determines their operational scale.